# 기후 커뮤니케이션 포럼 발제 — 3개 전국선거 기후·에너지 담론 비교 분석

**대상 선거(각 D-30 ~ D-1 윈도우)**
- 제22대 국회의원선거 (2024-04-10) → 수집기간 2024-03-11 ~ 04-09
- 제21대 대통령선거 (2025-06-03, 궐위 조기대선) → 2025-05-04 ~ 06-02
- 제9회 전국동시지방선거 (2026-06-03) → 2026-05-04 ~ 06-02

**데이터**: 빅카인즈에서 `(선거 키워드) AND (기후 키워드)` 로 수집한 .xlsx 3종 (19개 표준 칼럼)

**해석상 경계 (중요)**: 교집합 코퍼스이므로 *모든 기사가 기후를 1회 이상 언급*합니다.
따라서 "전체 선거보도 대비 기후 비중"(현저성)은 이 데이터로 측정할 수 없고,
"기후가 호명될 때의 *구조·프레임·감성·의미장*과 그 선거 간 변화"를 분석합니다.

**분석 순서**
0. 환경 설정 / 감성사전 다운로드
1. 로드 & 정제  2. 토큰화 설계  3. 기초 통계(A)  4. 프레임 어군 출현율(B)
5. LDA 토픽모델링(C)  6. 의미망 중심성(D)  7. 정파성 변별어(E)  8. 감성(F)  9. 통시 임베딩(G)
10. 종합·한계·다음 단계


## 0. 환경 설정

In [1]:
# 최초 1회만 실행 (이미 설치돼 있으면 건너뛰기)
# %pip install pandas openpyxl gensim scikit-learn networkx numpy

import re, json, numpy as np, pandas as pd
from collections import Counter
from itertools import combinations
import warnings; warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 160)

# ▼ 빅카인즈 xlsx 3종이 있는 폴더로 바꾸세요 (예: "/mnt/user-data/uploads")
DATA_DIR = "."
FILES = {"총선2024":"2024년총선.xlsx", "대선2025":"2025년대선.xlsx", "지선2026":"2026년지선.xlsx"}
ORDER = ["총선2024","대선2025","지선2026"]

### 0-1. 감성사전(KNU) 내려받기
KNU 한국어 감성사전(14,854개 표제어, 극성 −2~+2). 8단계 감성 분석에서 사용합니다.

In [2]:
import os, urllib.request
KNU_URL = "https://raw.githubusercontent.com/park1200656/KnuSentiLex/master/data/SentiWord_info.json"
if not os.path.exists("knu.json"):
    try:
        urllib.request.urlretrieve(KNU_URL, "knu.json")
        print("KNU 사전 다운로드 완료")
    except Exception as e:
        print("다운로드 실패 — 위 URL을 브라우저로 받아 knu.json 으로 같은 폴더에 두세요.\n", e)
else:
    print("knu.json 이미 존재")

KNU 사전 다운로드 완료


## 1. 데이터 로드 & 정제
'분석제외 여부' 칼럼의 **중복·예외** 행을 제거합니다.

In [3]:
data = {}
rows = []
for name in ORDER:
    df = pd.read_excel(os.path.join(DATA_DIR, FILES[name]))
    n_raw = len(df)
    df = df[df['분석제외 여부'].isna()].copy()      # 중복/예외 제거
    data[name] = df
    s = pd.to_datetime(df['일자'].astype(str), format='%Y%m%d')
    rows.append({"선거":name, "원본건수":n_raw, "유효건수":len(df),
                 "시작일":s.min().date(), "종료일":s.max().date(), "일평균":round(len(df)/30,1)})
summary = pd.DataFrame(rows).set_index("선거")
summary

,원본건수,유효건수,시작일,종료일,일평균
선거,,,,,
총선2024,2707,2625,2024-03-11,2024-04-09,87.5
대선2025,4052,3898,2025-05-04,2025-06-02,129.9
지선2026,3825,3753,2026-05-04,2026-06-02,125.1


## 2. 토큰화 설계
'키워드' 칼럼(전문 추출 명사)을 분석 단위로 씁니다. 불용어 철학:
- **제거**: 선거 절차어(후보·공약·유세…), 정당·인물명, 술어성 명사(강조·추진·마련…), 웹 스크랩/영문 노이즈
- **보존**: 프레임 변별력이 있는 **경제·산업·에너지·원전·재생·전환·AI** 어휘

인물명은 빅카인즈 '인물' 칼럼에서 동적으로 수집해 제거합니다.

In [4]:
STOP = set('''선거 후보 후보자 공약 정책 제시 약속 출마 유세 경선 표심 득표 당선 지지 지지율 여론조사 공약집
대선 총선 지선 지방선거 대통령선거 대통령 국회의원 시장 도지사 단체장 교육감 구청장 군수 시의원 도의원 의원
더불어민주당 국민의힘 민주당 국민 개혁신당 조국혁신당 정의당 진보당 대표 후보군 캠프 당 정당 여야 여당 야당
강조 추진 지원 확대 구축 대응 핵심 이날 조성 계획 강화 가능 마련 방안 상황 설명 육성 참여 확보 진행 시작
중심 분야 전략 대한 통해 위해 관련 지난 올해 기자 관계자 발표 입장 밝혀 예정 경우 지적 필요 중요 문제 대상
내용 이번 오늘 최근 현재 지역 국가 대한민국 정치 사회 국민들 국정 공동 전국 촉구 목표 방향 방침 제안 주장
다양 적극 실현 중점 집중 확정 결정 발언 모두 대해 라며 면서 한편'''.split())
NOISE = {"교황","콘클라베","추기경","레오","비엔나","바로가기","바로","가기","이메일","stating","adding",
         "abc","hl","www","com","co","kr","http","https","tv","candidate","party","kim","google","translate","local"}
NUM = re.compile(r'^[0-9][0-9.,%]*$')

def person_set():
    P=set()
    for name in ORDER:
        for cell in data[name]['인물'].fillna(''):
            for p in str(cell).split(','):
                p=p.strip()
                if len(p)>=2: P.add(p)
    return P
PERS = person_set()
DROP = PERS | NOISE
print("수집된 인물명:", len(PERS))

def clean(s, dedup=False, drop_names=True):
    """키워드 문자열 → 토큰 리스트. dedup=True면 문서 내 중복 제거(이진 출현용)."""
    out=[]
    for t in str(s).split(','):
        t=t.strip()
        if not t or t in STOP or NUM.match(t) or len(t)<2: continue
        if drop_names and t in DROP: continue
        out.append(t)
    if dedup:
        seen=set(); out=[x for x in out if not (x in seen or seen.add(x))]
    return out

# 동작 확인
print("예시 토큰:", clean(data['총선2024']['키워드'].iloc[0])[:15])

수집된 인물명: 7988
예시 토큰: ['기후', '사업', '남발', '시민단체', '전수조사', '지역구', '조사', '식량', '농업', '부족', '지역구', '기후', '비율', '기후위기', '의제']


## 3. 기초 통계 (A)
선거별 문서 수·토큰 규모·일자별 보도량.

In [5]:
rowsA=[]
for name in ORDER:
    docs=[clean(s) for s in data[name]['키워드'].fillna('')]; docs=[d for d in docs if d]
    ntok=sum(len(d) for d in docs)
    s=pd.to_datetime(data[name]['일자'].astype(str),format='%Y%m%d')
    daily=s.dt.date.value_counts()
    rowsA.append({"선거":name,"문서수":len(docs),"총토큰":ntok,"문서당평균토큰":round(ntok/len(docs)),
                  "일최대":f"{daily.idxmax()} ({daily.max()})","일최소":f"{daily.idxmin()} ({daily.min()})"})
pd.DataFrame(rowsA).set_index("선거")

,문서수,총토큰,문서당평균토큰,일최대,일최소
선거,,,,,
총선2024,2625,579827,221,2024-04-02 (165),2024-03-16 (15)
대선2025,3898,820624,211,2025-05-19 (261),2025-05-04 (9)
지선2026,3753,840511,224,2026-05-19 (238),2026-05-09 (30)


## 4. 핵심 프레임 어군 출현율 (B) — 통제 비교
이론 기반 어군을 정의하고, **각 선거에서 해당 어군 어휘를 1개 이상 포함한 문서 비율**을 비교합니다.
가장 직접적으로 연구 질문에 답하는 표입니다.

In [6]:
GROUPS={
 "기후 일반":{"기후","기후위기","기후변화","기후재난","폭염","이상기후"},
 "경제·산업":{"경제","산업","기업","성장","투자","금융","일자리","수출","경쟁력","반도체"},
 "재생에너지":{"재생에너지","재생","태양광","풍력","신재생","re100","해상풍력"},
 "에너지전환·전력":{"에너지전환","전환","전력","전력망","송전","전력수급","전기요금","전기료"},
 "원자력·원전":{"원전","원자력","smr","핵발전","핵발전소","두코바니","체르노빌","한수원"},
 "탄소중립·배출":{"탄소중립","온실가스","탄소","넷제로","감축","배출","배출량","탄소세","배출권"},
 "AI·데이터센터":{"ai","데이터센터","피지컬ai","인공지능"},
 "안보·자립":{"안보","에너지안보","자립","에너지자립","주권"},
 "제도·거버넌스":{"기본법","규제","기후에너지환경부","환경부","위원회"},
}
RAW={n:[set(clean(s,dedup=True,drop_names=False)) for s in data[n]['키워드'].fillna('')] for n in ORDER}
RAW={n:[d for d in RAW[n] if d] for n in RAW}
tbl={}
for g,terms in GROUPS.items():
    tbl[g]=[round(sum(1 for d in RAW[n] if terms & d)/len(RAW[n])*100,1) for n in ORDER]
frame_df=pd.DataFrame(tbl, index=ORDER).T
frame_df["변화(총선→지선)"]=(frame_df["지선2026"]-frame_df["총선2024"]).round(1)
frame_df
# 주: '정의로운 전환'은 단일 명사로 추출되지 않아 본문 구(句) 검색이 별도로 필요

,총선2024,대선2025,지선2026,변화(총선→지선)
기후 일반,60.3,43.1,37.9,-22.4
경제·산업,62.2,81.6,79.1,16.9
재생에너지,24.5,49.2,46.7,22.2
에너지전환·전력,26.4,44.7,41.4,15.0
원자력·원전,17.6,29.8,10.3,-7.3
탄소중립·배출,24.4,23.4,20.2,-4.2
AI·데이터센터,5.0,22.2,15.2,10.2
안보·자립,9.4,15.8,14.4,5.0
제도·거버넌스,31.7,38.5,31.2,-0.5


## 5. LDA 토픽모델링 (C)
**비교가능성**을 위해 3개 선거를 합친 통합 코퍼스에 단일 LDA를 학습시키고,
선거별 평균 토픽 비중을 비교합니다.

In [8]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 4.4 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]2m1/2 [gensim]
Note: you may need to restart the kernel to use updated packages.


In [9]:
from gensim import corpora
from gensim.models import LdaModel

docs=[]; labels=[]
for name in ORDER:
    for s in data[name]['키워드'].fillna(''):
        t=clean(s)
        if len(t)>=5: docs.append(t); labels.append(name)
labels=np.array(labels)
dic=corpora.Dictionary(docs); dic.filter_extremes(no_below=10, no_above=0.5)
bow=[dic.doc2bow(d) for d in docs]
K=12
lda=LdaModel(bow, num_topics=K, id2word=dic, passes=8, random_state=42, alpha='auto', eta='auto')

for k in range(K):
    print(f"T{k:02d}: " + ", ".join(w for w,_ in lda.show_topic(k, 10)))

T00: 호소, 투표, 생각, 사람, 위원장, 선택, 마지막, 운동, 경남, 보수
T01: 제주, 청년, 소득, 경제, 기본, 복지, 공공, 예산, 일자리, 인구
T02: 미국, 한국, 세계, 중국, 기후, 기업, 글로벌, 기준, 수준, 가격
T03: Candidate, 번역, Kim, Party, 영문, Google, Translate, Lee, future, Korea
T04: 교육, 학교, 국회, 여성, 비례, 노동자, 노동, 위원회, 학생, 교수
T05: 에너지, 원전, 전력, 데이터, 재생, AI, 산업, 태양광, 발전, 사업
T06: 기후, 전환, 시민, 환경, 충남, 에너지, 탄소, 기후위기, 중립, 생태
T07: 안전, 환경, 생활, 사업, 시설, 주민, 현장, 관리, 농업, 운영
T08: 경기도, 경제, 반도체, 경기, 지사, 평택, 기업, 규제, 용인, 외교
T09: 투자, 총리, 장관, 기업, 금융, AI, 전망, 반도체, 주가, 실적
T10: 산업, AI, 미래, 전북, 도시, 유치, 성장, 발전, 사업, 문화
T11: 의혹, 토론, 토론회, 조사, 비판, 생각, 의사, 제기, 보도, 사실


In [10]:
# 선거별 평균 토픽 비중(%)
theta=np.zeros((len(docs),K))
for i,b in enumerate(bow):
    for k,p in lda.get_document_topics(b, minimum_probability=0): theta[i,k]=p
topic_share=pd.DataFrame({n:(theta[labels==n].mean(0)*100).round(1) for n in ORDER})
topic_share.index=[f"T{k:02d} ("+", ".join(w for w,_ in lda.show_topic(k,3))+")" for k in range(K)]
topic_share.sort_values("지선2026", ascending=False)

,총선2024,대선2025,지선2026
"T10 (산업, AI, 미래)",9.3,11.6,24.1
"T01 (제주, 청년, 소득)",9.0,8.0,15.4
"T00 (호소, 투표, 생각)",20.2,13.7,11.2
"T07 (안전, 환경, 생활)",6.4,4.1,11.2
"T06 (기후, 전환, 시민)",7.8,9.0,8.5
"T04 (교육, 학교, 국회)",14.6,5.4,7.1
"T11 (의혹, 토론, 토론회)",8.1,9.2,5.5
"T05 (에너지, 원전, 전력)",4.5,14.0,5.3
"T02 (미국, 한국, 세계)",9.7,10.2,3.8
"T08 (경기도, 경제, 반도체)",5.7,7.6,3.8


## 6. 의미망 중심성 & 군집 (D)
선거별로 상위 빈도어 간 **문서 공출현** 네트워크를 만들고, 가중 연결중심성과 군집을 봅니다.
'기후'·'원전'의 망 내 위치 변화에 주목합니다.

In [11]:
import networkx as nx
for name in ORDER:
    cdocs=[clean(s,dedup=True) for s in data[name]['키워드'].fillna('')]; cdocs=[d for d in cdocs if d]
    df_=Counter()
    for d in cdocs: df_.update(d)
    top=set(t for t,_ in df_.most_common(140))
    co=Counter()
    for d in cdocs:
        pr=sorted(t for t in set(d) if t in top)
        for a,b in combinations(pr,2): co[(a,b)]+=1
    G=nx.Graph()
    for (a,b),w in co.items():
        if w>=8: G.add_edge(a,b,weight=w)
    wd=sorted(G.degree(weight='weight'), key=lambda x:-x[1])[:12]
    print(f"\n[{name}] 가중연결도 Top12: " + ", ".join(t for t,_ in wd))
    for key in ["기후","원전"]:
        if key in G:
            nb=sorted(G[key].items(), key=lambda x:-x[1]['weight'])[:6]
            print(f"   · '{key}' 인접: " + ", ".join(n for n,_ in nb))
    comms=sorted(nx.community.greedy_modularity_communities(G, weight='weight'), key=len, reverse=True)[:3]
    for i,c in enumerate(comms):
        cl=sorted(c, key=lambda n:-G.degree(n,weight='weight'))[:7]
        print(f"   군집 C{i}: " + ", ".join(cl))


[총선2024] 가중연결도 Top12: 기후, 미래, 경제, 발전, 산업, 국회, 해결, 에너지, 환경, 서울, 기업, 생각
   · '기후' 인접: 미래, 국회, 환경, 경제, 에너지, 발전
   · '원전' 인접: 발전, 산업, 생각, 미래, 경제, 비판
   군집 C0: 경제, 발전, 산업, 해결, 에너지, 환경, 기업
   군집 C1: 기후, 미래, 국회, 서울, 생각, 대책, 위원장

[대선2025] 가중연결도 Top12: 산업, 에너지, 경제, 발전, 기업, AI, 개혁, 전환, 미래, 기후, 성장, 재생
   · '기후' 인접: 에너지, 산업, 전환, 개혁, 경제, 환경
   · '원전' 인접: 산업, 에너지, AI, 발전, 기업, 경제
   군집 C0: 산업, 에너지, 경제, 발전, 기업, AI, 전환
   군집 C1: 개혁, 기후, 서울, 평가, 생각, 위원회, 언급

[지선2026] 가중연결도 Top12: 산업, 미래, 경제, 발전, 에너지, 사업, AI, 도시, 성장, 전환, 유치, 행정
   · '기후' 인접: 환경, 미래, 산업, 지방, 에너지, 경제
   군집 C0: 발전, 에너지, 사업, 전환, 행정, 체계, 구조
   군집 C1: 산업, 미래, 경제, AI, 성장, 유치, 청년
   군집 C2: 도시, 시민, 현장, 변화, 해결, 단순, 평가


## 7. 정파성 변별어 (E)
언론사를 정파·유형군으로 묶고, **보수지 vs 진보지**의 어휘를 log-odds로 비교합니다.

In [12]:
MG={**{m:"보수지" for m in ["조선일보","중앙일보","동아일보","문화일보","세계일보","데일리안"]},
    **{m:"진보지" for m in ["한겨레","경향신문","프레시안","오마이뉴스","미디어오늘"]},
    **{m:"경제지" for m in ["매일경제","한국경제","서울경제","머니투데이","이데일리","아시아경제","파이낸셜뉴스","헤럴드경제","이투데이","아주경제","뉴스핌"]},
    **{m:"방송" for m in ["KBS","MBC","SBS","YTN","JTBC"]}}
def grp(m): return MG.get(str(m).strip(),"기타/지역")

# 매체군별 기사 수
gc=Counter()
for name in ORDER:
    for m in data[name]['언론사'].fillna(''): gc[grp(m)]+=1
print("매체군별 기사 수:", dict(gc.most_common()))

매체군별 기사 수: {'기타/지역': 5778, '경제지': 2377, '진보지': 798, '보수지': 730, '방송': 593}


In [13]:
cons=Counter(); prog=Counter(); Nc=Np=0
SELF={"데일리안","뉴시스","경향신문","한겨레","연합뉴스"}   # 매체 자기참조 제거
for name in ORDER:
    for _,row in data[name].iterrows():
        g=grp(row['언론사']); t=set(clean(row['키워드'],dedup=True))-SELF
        if g=="보수지": cons.update(t); Nc+=1
        elif g=="진보지": prog.update(t); Np+=1
vocab=[t for t in (cons|prog) if cons[t]+prog[t]>=15]
sc=sorted(((np.log2((cons[t]/Nc+1e-3)/(prog[t]/Np+1e-3)), t) for t in vocab), reverse=True)
print("[보수지 우세]", ", ".join(t for _,t in sc[:15]))
print("[진보지 우세]", ", ".join(t for _,t in sc[-15:][::-1]))

[보수지 우세] 한미, 왼쪽, 영토, 디자인, 허위, 공조, 조롱, 두산에너빌리티, 아프리카, 소방, 상장, 청문회, 전기료, 셰셰, 10조
[진보지 우세] 진보정당, 티브이, 핵발전, 진보정치, 소수자, 핵폐기물, RE, 핵발전소, 이주노동자, 비상행동, 차별금지, 성소수자, 녹색당, 사회대개혁, 농민들


## 8. 감성 분석 (F)
KNU 사전을 '키워드' 명사에 매칭해 문서 평균 극성을 계산합니다.
**한계**: 명사 기반이라 부정어·서술어 극성은 누락 → *상대 비교용*으로만 해석.

In [17]:
knu = json.load(open("knu.json"))
POL = {}
for e in knu:
    try: p = int(e['polarity'])
    except: continue
    POL[e['word']] = p
    if e.get('word_root'): POL.setdefault(e['word_root'], p)

def senti(toks):
    v = [POL[t] for t in toks if t in POL]
    return np.mean(v) if v else 0.0

# 매체군(행) × 선거(열) 평균 감성
groups = ["보수지","진보지","경제지","방송","기타/지역"]
mat = {}
for grp_name in groups:
    mat[grp_name] = []
    for name in ORDER:
        ss = [senti(clean(row['키워드'], drop_names=False))
              for _, row in data[name].iterrows() if grp(row['언론사']) == grp_name]
        mat[grp_name].append(round(np.mean(ss), 3) if ss else np.nan)
senti_df = pd.DataFrame(mat, index=ORDER).T          # 5 매체군 × 3 선거

# 선거별 '전체' 평균을 '행'으로 추가 (열 개수=3 과 일치하므로 오류 없음)
overall = {n: round(np.mean([senti(clean(s, drop_names=False)) for s in data[n]['키워드'].fillna('')]), 3) for n in ORDER}
senti_df.loc["전체"] = [overall[n] for n in ORDER]

senti_df

,총선2024,대선2025,지선2026
보수지,0.034,0.023,0.125
진보지,-0.125,-0.053,0.170
경제지,0.132,0.230,0.265
방송,-0.138,-0.025,0.070
기타/지역,0.130,0.164,0.281
전체,0.086,0.141,0.254


## 9. 통시적 Word2Vec (G)
선거별로 임베딩을 **독립 학습**하고, 핵심어의 최근접 이웃을 비교합니다.
**주의**: 모델이 분리돼 있어 모델 간 코사인 값은 비교 불가 → *이웃 집합(set)* 의 변화만 해석.

In [15]:
from gensim.models import Word2Vec
M={}
for name in ORDER:
    sents=[clean(s) for s in data[name]['키워드'].fillna('')]; sents=[x for x in sents if len(x)>=5]
    M[name]=Word2Vec(sents, vector_size=80, window=6, min_count=12, workers=4, sg=1, epochs=10, seed=42)
    print(name, "어휘수:", len(M[name].wv))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


총선2024 어휘수: 6237


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


대선2025 어휘수: 7480


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


지선2026 어휘수: 7348


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [16]:
for key in ["기후","원전","재생에너지","전환","탄소중립"]:
    print(f"\n── '{key}' 최근접 이웃 ──")
    for name in ORDER:
        wv=M[name].wv
        if key in wv:
            print(f"  {name}: " + ", ".join(w for w,_ in wv.most_similar(key, topn=8)))
        else:
            print(f"  {name}: (어휘 없음)")


── '기후' 최근접 이웃 ──
  총선2024: 기후위기, 기후정치, 기후공약, 기후대응, 탄소감축, 기후패스, 최우선, 녹색
  대선2025: 기후위기, 기후정책, 기후대응, 기후재난, 사회정의, 현세대, 환경문제, 적응
  지선2026: 기후위기, 적응, 기후재난, 기후정책, 의제, 이상기후, 로컬에너지랩, 부소장

── '원전' 최근접 이웃 ──
  총선2024: 원전산업, SMR, 핵발전, 원자력, 영구, 소형모듈원전, 태양광업자들, 원자로
  대선2025: 탈원전, 소형모듈원전, SMR, 6기, 원자력, 세일즈, 원전산업, 소형모듈
  지선2026: SMR, 원자력, 소형모듈원자로, 한국수력원자력, 소형모듈원전, 방폐장, 원자로, 한수원

── '재생에너지' 최근접 이웃 ──
  총선2024: 재생, 에너지, CFE, 신재생에너지, RE100, 균형적, 원자력발전, 에너지원
  대선2025: 재생, 에너지, 서남해안, 무탄소에너지, 기저전력, 살길, 에너지믹스, 에너지원
  지선2026: 재생, 에너지, RE100, 지산지소, 그린수소, 산업용지, 에너지저장장치, 출력제어

── '전환' 최근접 이웃 ──
  총선2024: 에너지전환, 산업전환, 정의, 기후대응, 일터, 탈핵, 탄소감축, 공공재생에너지
  대선2025: 탈탄소화, 산업전환, 에너지, 저탄소, 탈탄소, 탄소중립, 상용차, 그린수소
  지선2026: 탈피, 산업, 석탄발전, 국제고, 외고, 일반고, 대입자격고사, 산업구조

── '탄소중립' 최근접 이웃 ──
  총선2024: 탄소감축, 탄소, 중립, 국가경쟁력, 탄소중립도시, NDC, 과천시, 저탄소
  대선2025: 중립, 탄소, 녹색성장, 기술혁신, 축산분야, 대통령직속, 기업가정신, 산업구조
  지선2026: 탄소, 중립, 녹색성장, 에너지전환, 자원순환, 실천, 흡수원, 농축산


## 10. 종합 · 한계 · 다음 단계

**수렴하는 발견**
- *양적 위축·탈중심화*: 일반 '기후' 프레임 60→44→38%↓ (B), 순수 기후토픽 T03 ~10% 정체 (C), '기후' 의미망 중심성 1위→주변 (D)
- *질적 재맥락화*: 경제·산업 프레임 62→82→79% (B), 숙주 토픽 이동 = 총선 의정기제 → 대선 원전·에너지안보·AI투자 → 지선 지역개발 (C), 의미장 이동 (G)
- *정파 분극화*: 원전(보수: 두산에너빌리티·전기료·산업) vs 핵발전(진보: 핵폐기물·녹색당·정의의제) (E), 진보지 비판톤 vs 경제지 기회톤 (F)

**한계**
1. 교집합 코퍼스 → 현저성(전체 대비 비중) 측정 불가 (별도 베이스라인 필요)
2. 감성: 명사 기반 사전 매칭, 부정어 미처리 (상대 비교용)
3. Word2Vec: 모델 독립 학습 → 이웃 집합만 비교
4. '전환' 임베딩에 교육(외고·국제고 전환) 다의어, 본문 일부 영문 스크랩 노이즈 잔존
5. 토픽 수 K=12는 한 선택 → K 스윕 강건성 점검 권장

**다음 단계 후보**
- 포스터용 시각화(프레임 추이·토픽 비중·의미망; 한글 폰트 적용)
- '정의로운 전환' 등 구(句) 단위 본문 200자 정규식 계량
- 비선거 베이스라인 수집 후 현저성 보정
- 공식 선거운동기 하위구간 강건성 검증
